In [ ]:
import tempfile
from pathlib import Path
from datasets import DatasetDict, load_from_disk
from huggingface_hub import list_bucket_tree, download_bucket_files

BUCKET = "avgJo3/final-mdlm-sft-artifacts"

PREFIXES: dict[str, str] = {
    "base":      "20260714T050657-0e2e053501df5170/data_",
    "reasoning": "20260715T045644-7d0f95338c3101b3/data_",   #
}

CHAIN_STEMS = {
    "train":        "train",          # (γ=strict, σ=reload)
    "mix":          "mix",            # (γ=mix,    σ=reload)
    "ablation":     "ablation",       # (γ=strict, σ=continue)
    "mix-ablation": "mix-ablation",   # (γ=mix,    σ=continue)
}
ROUNDS = ["R0", "R1", "R2", "R3", "R4"]

# Retained tempdirs, keyed by checkpoint alias, so lazy Dataset reads stay valid.
_TEMPDIRS: dict[str, tempfile.TemporaryDirectory] = {}


def load_chains_for(ckpt_alias: str) -> dict[str, DatasetDict]:
    if ckpt_alias not in PREFIXES:
        raise KeyError(f"unknown checkpoint alias {ckpt_alias!r}; "
                       f"known aliases: {list(PREFIXES)}")
    prefix = PREFIXES[ckpt_alias]
    if prefix is None:
        raise ValueError(f"prefix for {ckpt_alias!r} is not yet configured")

    files = [
        item for item in list_bucket_tree(BUCKET, prefix=prefix, recursive=True)
        if item.type == "file"
    ]
    if not files:
        raise RuntimeError(f"no files found under prefix: {prefix}")

    tmp = tempfile.TemporaryDirectory(prefix=f"chains-{ckpt_alias}-")
    _TEMPDIRS[ckpt_alias] = tmp
    tmp_root = Path(tmp.name)

    downloads = [(f, str(tmp_root / f.path)) for f in files]
    download_bucket_files(BUCKET, files=downloads)

    data_root = tmp_root / prefix.rstrip("/").removesuffix("data_").rstrip("/")
    shared_r0_dir = data_root / "data_train-generated_r0"
    assert shared_r0_dir.exists(), f"shared R0 dir not found: {shared_r0_dir}"

    def load_chain(stem: str) -> DatasetDict:
        splits = {"R0": load_from_disk(str(shared_r0_dir))}
        for k in range(1, len(ROUNDS)):
            round_dir = data_root / f"data_{stem}-generated_r{k}"
            splits[f"R{k}"] = load_from_disk(str(round_dir))
        return DatasetDict(splits)

    return {cid: load_chain(stem) for cid, stem in CHAIN_STEMS.items()}


# --- usage: load one checkpoint's chains -------------------------------------
chains_by_ckpt: dict[str, dict[str, DatasetDict]] = {}
chains_by_ckpt["base"] = load_chains_for("base")
chains_by_ckpt["reasoning"] = load_chains_for("reasoning")

print(chains_by_ckpt["base"])
print(chains_by_ckpt["reasoning"])

In [ ]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer("BAAI/bge-m3")
def embed_fn(batch):
    return {
        "embeddings": model.encode(
            batch["completion"],
            show_progress_bar=False,
            normalize_embeddings=True,
        )
    }

for ckpt_alias, chains in chains_by_ckpt.items():
    for chain_id, chain in chains.items():
        print(f"[embed] {ckpt_alias}/{chain_id}")
        chains_by_ckpt[ckpt_alias][chain_id] = chain.map(embed_fn, batched=True)

del model

In [ ]:
from datasets import Dataset, DatasetDict

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import wasserstein_distance


# --- per-round atomic function -----------------------------------------------
from sklearn.metrics.pairwise import cosine_similarity

def atomic_round(H: np.ndarray) -> dict:
    H = np.asarray(H, dtype=np.float32)
    n = H.shape[0]

    S = cosine_similarity(H)   # handles normalisation internally

    iu, ju = np.triu_indices(n, k=1)
    u = S[iu, ju]

    return {
        "s_bar":   float(u.mean()),
        "U_hat":   np.sort(u),
        "n":       n,
        "n_pairs": u.size,
    }


def process_chain(chain) -> tuple[pd.DataFrame, dict]:
    round_keys = [f"R{k}" for k in range(len(chain))]
    K = len(round_keys) - 1

    # atomic tier
    records = []
    U_hats  = {}
    for key in round_keys:
        H_k = np.asarray(chain[key]["embeddings"], dtype=np.float32)
        out = atomic_round(H_k)
        records.append({
            "round":   key,
            "n":       out["n"],
            "n_pairs": out["n_pairs"],
            "s_bar":   out["s_bar"],
        })
        U_hats[key] = out["U_hat"]

    df = pd.DataFrame(records).set_index("round")

    # intra-chain: scalar deltas
    df["delta_s_slide"] = df["s_bar"].diff()
    df["delta_s_orig"]  = df["s_bar"] - df["s_bar"].iloc[0]

    # intra-chain: W_1 deltas
    delta_U_slide = [np.nan]
    for k in range(1, K + 1):
        delta_U_slide.append(
            wasserstein_distance(U_hats[round_keys[k - 1]], U_hats[round_keys[k]])
        )

    delta_U_orig = [0.0]
    u_origin = U_hats[round_keys[0]]
    for k in range(1, K + 1):
        delta_U_orig.append(wasserstein_distance(u_origin, U_hats[round_keys[k]]))

    df["delta_U_slide"] = delta_U_slide
    df["delta_U_orig"]  = delta_U_orig

    # --- normalised drift path  phi_k = W1(R0, Rk) / W1(R0, RK) ------------
    total_drift = delta_U_orig[-1]          # scalar: W1(R0, RK)
    with np.errstate(divide="ignore", invalid="ignore"):
        df["phi"] = np.where(
            total_drift > 0,
            df["delta_U_orig"].to_numpy() / total_drift,
            np.nan,                         # chain never moved at all
        )

    # cross-object gap ratios
    with np.errstate(divide="ignore", invalid="ignore"):
        df["rho_slide"] = np.where(
            df["delta_U_slide"].to_numpy() > 0,
            np.abs(df["delta_s_slide"].to_numpy()) / df["delta_U_slide"].to_numpy(),
            np.nan,
        )
        df["rho_orig"] = np.where(
            df["delta_U_orig"].to_numpy() > 0,
            np.abs(df["delta_s_orig"].to_numpy()) / df["delta_U_orig"].to_numpy(),
            np.nan,
        )

    return df, U_hats


import pandas as pd
per_chain_dfs: dict[str, pd.DataFrame] = {}
per_chain_Uhats: dict[str, dict] = {}

for ckpt_alias, chains in chains_by_ckpt.items():
    for chain_id, chain in chains.items():
        key = f"{ckpt_alias}/{chain_id}"
        print(f"processing {key} ...")
        df, U_hats = process_chain(chain)
        df = df.assign(
            chain=key,
            round=range(len(df)),        # ← R0, R1, R2 ...
        )
        per_chain_dfs[key] = df
        per_chain_Uhats[key] = U_hats


all_df = pd.concat(per_chain_dfs.values(), ignore_index=True)
all_df.insert(0, "round", all_df.pop("round"))

In [ ]:
LATEX_PATH = "per_chain_summary.tex"

# --- clean up columns --------------------------------------------------------
display_df = (
    all_df
    .drop(columns=["n", "n_pairs"], errors="ignore")
)

# --- numeric formatting ------------------------------------------------------
float_cols = display_df.select_dtypes(include="float").columns
display_df[float_cols] = display_df[float_cols].map(lambda x: f"{x:.3f}")

# --- render to latex ---------------------------------------------------------
latex_str = display_df.to_latex(
    index=False,
    escape=True,
    column_format="l" + "r" * (len(display_df.columns) - 1),
    caption="Per-chain summary statistics across rounds.",
    label="tab:per_chain_summary",
)

with open(LATEX_PATH, "w") as f:
    f.write(latex_str)

print(latex_str)



In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np

# ── thesis style ──────────────────────────────────────────────────────────────
mpl.rcParams.update({
    "figure.facecolor":      "white",
    "figure.dpi":            300,
    "savefig.dpi":           300,
    "savefig.facecolor":     "white",
    "axes.facecolor":        "#f9f9f9",
    "axes.edgecolor":        "#cccccc",
    "axes.labelcolor":       "#222222",
    "axes.titlecolor":       "#111111",
    "axes.titlesize":        10,
    "axes.labelsize":        8,
    "axes.grid":             True,
    "axes.grid.axis":        "y",
    "axes.spines.top":       False,
    "axes.spines.right":     False,
    "grid.color":            "#e0e0e0",
    "grid.linestyle":        "--",
    "grid.linewidth":        0.6,
    "xtick.color":           "#444444",
    "ytick.color":           "#444444",
    "xtick.labelsize":       7,
    "ytick.labelsize":       7,
    "legend.facecolor":      "white",
    "legend.edgecolor":      "#cccccc",
    "legend.labelcolor":     "#222222",
    "legend.fontsize":       7,
    "legend.title_fontsize": 7,
    "lines.linewidth":       1.5,
    "lines.markersize":      3,
    "font.family":           "sans-serif",
    "font.sans-serif":       ["Inter", "Helvetica Neue", "Arial", "DejaVu Sans"],
    "text.color":            "#222222",
})
THESIS_COLORS = [
    "#0072B2",   # blue
    "#B22222",   # firebrick red
]

ALPHA_IQR  = 0.15
ROW_GROUPS = [["train", "mix"], ["ablation", "mix-ablation"]]
COL_CKPTS  = ["base", "reasoning"]
ROW_LABELS = ["train  ·  mix", "ablation  ·  mix-ablation"]

fig, axes = plt.subplots(
    2, 2,
    figsize=(6, 4),
    sharey=True,
    constrained_layout=False,
)
fig.subplots_adjust(bottom=0.12, hspace=0.38, wspace=0.08)

for row_idx, (chain_ids, row_label) in enumerate(zip(ROW_GROUPS, ROW_LABELS)):
    for col_idx, ckpt_alias in enumerate(COL_CKPTS):

        ax      = axes[row_idx, col_idx]
        keys    = [f"{ckpt_alias}/{cid}" for cid in chain_ids]
        handles = []
        labels  = []

        for k_idx, key in enumerate(keys):
            if key not in per_chain_dfs:
                continue

            color      = THESIS_COLORS[k_idx % len(THESIS_COLORS)]
            df         = per_chain_dfs[key]
            U_hats     = per_chain_Uhats[key]
            round_keys = list(U_hats.keys())
            x          = np.arange(len(round_keys))
            s_bar      = df["s_bar"].values

            p5, p25, p75, p95 = zip(*[
                np.percentile(U_hats[rk], [5, 25, 75, 95])
                for rk in round_keys
            ])
            p5, p25, p75, p95 = map(np.array, (p5, p25, p75, p95))

            chain_id = key.split("/", 1)[1]

            # 5–95: dashed boundary lines
            ax.plot(x, p5,  color=color, linewidth=0.8, linestyle="--", alpha=0.5, zorder=2)
            ax.plot(x, p95, color=color, linewidth=0.8, linestyle="--", alpha=0.5, zorder=2)

            # IQR fill
            ax.fill_between(x, p25, p75, color=color, alpha=ALPHA_IQR, linewidth=0, zorder=1)

            # mean line
            line, = ax.plot(x, s_bar,
                            color=color, linewidth=2.5, marker="o", markersize=5,
                            markeredgecolor="white", markeredgewidth=0.8,
                            zorder=3, label=chain_id)
            handles.append(line)
            labels.append(chain_id)

        if row_idx == 0:
            ax.set_title(ckpt_alias, fontweight="bold")
        ax.set_xlabel("Round")
        ax.set_xlim(0, len(round_keys) - 1)
        ax.set_xticks(x)
        ax.set_xticklabels(round_keys)

        # legend beneath the panel
        ax.legend(
            handles, labels,
            title="chain",
            loc="upper center",
            bbox_to_anchor=(0.5, -0.18),
            ncol=len(handles),
            framealpha=0.6,
        )

        if col_idx == 0:
            ax.set_ylabel(f"{row_label}\n\npairwise cosine similarity")

fig.suptitle(
    "Semantic drift per model  ·  line = s̄   ·  bands = IQR / 5–95th pct",
    fontsize=12,
)
#plt.savefig("semantic_drift.png", bbox_inches="tight")
#plt.show()

In [ ]:
from itertools import combinations
import numpy as np
import pandas as pd
from scipy.stats import wasserstein_distance

def cross_pair_long(key_a: str, key_b: str) -> list[dict]:
    df_a = per_chain_dfs[key_a]
    df_b = per_chain_dfs[key_b]
    if not df_a.index.equals(df_b.index):
        raise ValueError(f"round misalignment: {key_a} vs {key_b}")

    rows = []
    for r in df_a.index:
        d_bar = float(df_a.loc[r, "s_bar"] - df_b.loc[r, "s_bar"])
        w1    = wasserstein_distance(
                    per_chain_Uhats[key_a][r],
                    per_chain_Uhats[key_b][r],
                )
        rows.append({
            "chain_A":    key_a,
            "chain_B":    key_b,
            "ckpt_A":     key_a.split("/", 1)[0],
            "ckpt_B":     key_b.split("/", 1)[0],
            "chain_id_A": key_a.split("/", 1)[1],
            "chain_id_B": key_b.split("/", 1)[1],   # fix
            "round":      r,
            "d_bar":      d_bar,
            "W1":         w1,
            "rho":        abs(d_bar) / w1 if w1 > 0 else np.nan,
        })
    return rows

chain_keys = [
    f"{ckpt_alias}/{chain_id}"
    for ckpt_alias, chains in chains_by_ckpt.items()
    for chain_id in chains.keys()
]

all_pairs = []
for key_a, key_b in combinations(chain_keys, 2):   # n(n-1)/2 instead of n²
    print(f"cross-pair: {key_a}  ×  {key_b}")
    all_pairs.extend(cross_pair_long(key_a, key_b))

pair_long = pd.DataFrame(all_pairs)

n_rounds   = len(next(iter(per_chain_dfs.values())))
n_pairs    = len(chain_keys) * (len(chain_keys) - 1) // 2
print(f"\npair_long rows: {len(pair_long)}  "
      f"(expected {n_pairs} × {n_rounds} = {n_pairs * n_rounds})")

# --- sanity checks — adapted for upper-triangle only ------------------------
sw = pair_long.merge(
    pair_long,
    left_on =["chain_A", "chain_B", "round"],
    right_on=["chain_B", "chain_A", "round"],
    suffixes=("", "_sw"),
)
# with combinations there is no diagonal and no swapped duplicate,
# so the merge above will be empty — just verify W1 >= 0 and rho >= 0
assert (pair_long["W1"]  >= 0).all(),  "W1 must be non-negative"
assert (pair_long["rho"].dropna() >= 0).all(), "rho must be non-negative"

print("\nsanity checks passed ✓")
print(pair_long.head(10))

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
import seaborn as sns

ROUNDS     = sorted(pair_long["round"].unique())
COL_CKPTS  = ["base", "reasoning"]

# per-ckpt chain keys and labels
ckpt_chain_keys = {
    ckpt: [k for k in per_chain_dfs.keys() if k.startswith(f"{ckpt}/")]
    for ckpt in COL_CKPTS
}
ckpt_labels = {
    ckpt: [k.split("/")[1] for k in keys]
    for ckpt, keys in ckpt_chain_keys.items()
}

# shared colour scale across both ckpts and all rounds
vabs   = float(np.nanmax(np.abs(pair_long["d_bar"].values)))
W1_max = float(np.nanmax(pair_long["W1"].values))
norm_d = mcolors.TwoSlopeNorm(vmin=-vabs, vcenter=0, vmax=vabs)

def pivot_round(round_key, value_col, chain_keys):
    sub = pair_long[pair_long["round"] == round_key]
    return (
        sub.pivot(index="chain_A", columns="chain_B", values=value_col)
           .reindex(index=chain_keys, columns=chain_keys)
           .to_numpy(dtype=float)
    )

n_rounds = len(ROUNDS)

# 2 rows (ckpt) × n_rounds cols
fig, axes = plt.subplots(
    2, n_rounds,
    figsize=(4.2 * n_rounds, 4.2 * 2),
    constrained_layout=True,
)

for row_idx, ckpt in enumerate(COL_CKPTS):
    chain_keys = ckpt_chain_keys[ckpt]
    labels     = ckpt_labels[ckpt]
    n          = len(chain_keys)
    mask       = np.tril(np.ones((n, n), dtype=bool))

    for col_idx, round_key in enumerate(ROUNDS):
        ax  = axes[row_idx, col_idx]
        M_d = pivot_round(round_key, "d_bar", chain_keys)
        M_w = pivot_round(round_key, "W1",    chain_keys)

        show_cbar = (col_idx == n_rounds - 1) and (row_idx == 1)

        sns.heatmap(
            M_d, ax=ax,
            cmap="RdBu", norm=norm_d,
            mask=mask,
            xticklabels=labels, yticklabels=labels,
            linewidths=0.5, linecolor="white",
            cbar=show_cbar,
            cbar_kws={"label": "d_bar (signed)", "shrink": 0.7},
            square=True,
            annot=False,
        )

        # cell annotations: d_bar text + W1 circle
        for i in range(n):
            for j in range(n):
                if mask[i, j]:
                    continue
                d = M_d[i, j]
                w = M_w[i, j]
                if np.isnan(d) or np.isnan(w):
                    continue
                radius = 0.35 * (w / W1_max) ** 0.5
                ax.add_patch(mpatches.Circle(
                    (j + 0.5, i + 0.5), radius,
                    color="black", alpha=0.18, zorder=3,
                ))
                ax.text(
                    j + 0.5, i + 0.5, f"{d:+.3f}",
                    ha="center", va="center",
                    fontsize=6.5, color="#111111", zorder=4,
                )

        # titles and labels
        if row_idx == 0:
            ax.set_title(round_key, fontweight="bold", fontsize=10)
        ax.tick_params(axis="x", rotation=45, labelsize=7)
        ax.tick_params(axis="y", rotation=0,  labelsize=7)

        # suppress redundant y labels
        if col_idx > 0:
            ax.set_ylabel("")
            ax.set_yticklabels([])

    # row label on leftmost panel
    axes[row_idx, 0].set_ylabel(f"{ckpt}\n\nchain B", fontsize=8)

# row annotations on the right
for row_idx, ckpt in enumerate(COL_CKPTS):
    axes[row_idx, -1].annotate(
        ckpt, xy=(1.02, 0.5), xycoords="axes fraction",
        fontsize=9, fontweight="bold", va="center", rotation=-90,
    )

fig.suptitle(
    "Pairwise cross-chain d_bar per round  ·  "
    "colour = d_bar (RdBu)  ·  circle size = W1",
    fontsize=11,
)
plt.savefig("pairwise_heatmap.png", bbox_inches="tight")
plt.show()